# TP Final Integrador: Análisis y Optimización del Negocio con Python y pandas

**Dataset:** Proyecto III – Todos al Agua

**Integrantes:** Lisandro (Fase 1: diagnóstico, limpieza y ETL) · Fragiee (Fase 2–3: estadística, IQR y visualización). Fase 4 y este notebook: trabajo conjunto.

**Objetivo:** recorrer el flujo completo de trabajo de un consultor de datos: diagnosticar la calidad, limpiar y transformar, explorar y visualizar, y finalmente fundamentar estrategias de negocio accionables.

Este notebook usa el CSV limpio generado en `fase_1/proyecto_3_todos_al_agua_limpio.csv`. No modifica los archivos de la carpeta `fase_2/`.


## Fase 1 – Diagnóstico e inspección inicial

Carga del archivo crudo, análisis estructural y auditoría de calidad (nulos y duplicados exactos).


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / 'proyecto_3_todos_al_agua_crudo.csv').exists():
    RAIZ = RAIZ.parent
CRUDO = RAIZ / 'proyecto_3_todos_al_agua_crudo.csv'
LIMPIO = RAIZ / 'fase_1' / 'proyecto_3_todos_al_agua_limpio.csv'
FIGURAS = RAIZ / 'informe'
FIGURAS.mkdir(exist_ok=True)

crudo = pd.read_csv(CRUDO)
crudo.info()
print('Dimensiones:', crudo.shape)

nulos = crudo.isna().sum()
print('\nNulos por columna:')
print(nulos[nulos > 0])
print('Celdas nulas:', int(nulos.sum()), 'de', crudo.size,
      f'({100 * nulos.sum() / crudo.size:.2f}%)')
print('Filas con nulos:', int(crudo.isna().any(axis=1).sum()))

print('\nDuplicados exactos:', int(crudo.duplicated().sum()))
print(crudo[crudo.duplicated(keep='first')])


## Fase 2 – Limpieza y transformación (ETL)

Criterios aplicados:

1. Se eliminan los 8 duplicados exactos (IDs 3081–3088).
2. Los nulos de texto pasan a `'Sin dato'`; se conservan las filas porque son solo el 1,28% de las celdas.
3. Las categorías se normalizan (`Sí/No`, `Masculino/Femenino/No Binario`) para que los `groupby` no separen etiquetas equivalentes.
4. Los dos valores inválidos (−150.000 y 15.000.000) se marcan y se imputan con la mediana de su grupo (actividad / sector), no con cero.
5. Se crea `rango_etario` y se consolidan métricas con `groupby().agg()`.


In [ ]:
df = crudo.copy()
df = df.drop_duplicates(keep='first', ignore_index=True)

df['sectores'] = df['sectores'].fillna('Sin dato')
df['tipo_discapacidad'] = df['tipo_discapacidad'].fillna('Sin dato')

MAPA_CUD = {'no': 'No', 'si': 'Sí', 'sí': 'Sí', 'tiene cud': 'Sí'}
df['tiene_cud'] = (df['tiene_cud'].astype('string').str.strip().str.lower()
                   .map(MAPA_CUD).fillna('Sin dato'))

MAPA_SEXO = {'m': 'Masculino', 'masculino': 'Masculino',
             'f': 'Femenino', 'femenino': 'Femenino',
             'no binario': 'No Binario'}
df['identidad_sexual'] = (df['identidad_sexual'].astype('string').str.strip().str.lower()
                          .map(MAPA_SEXO).fillna('Sin dato'))

df['salario_original'] = df['salario'].astype('string').fillna('Sin dato')
df['ganancias_original'] = df['ganancias_sector'].copy()

mascara_salario = df['salario'].isna() | (df['salario'] <= 0)
df['salario_imputado'] = mascara_salario
salario_valido = df['salario'].where(~mascara_salario)
mediana_actividad = salario_valido.groupby(df['actividades']).transform('median')
df.loc[mascara_salario, 'salario'] = mediana_actividad[mascara_salario]
df['salario'] = df['salario'].fillna(salario_valido.median())

mascara_ganancias = df['ganancias_sector'] > 1_000_000
df['ganancias_imputada'] = mascara_ganancias
ganancias_validas = df['ganancias_sector'].where(~mascara_ganancias)
mediana_sector = ganancias_validas.groupby(df['sectores']).transform('median')
df.loc[mascara_ganancias, 'ganancias_sector'] = mediana_sector[mascara_ganancias]
df['ganancias_sector'] = df['ganancias_sector'].fillna(ganancias_validas.median())

df['rango_etario'] = pd.cut(
    df['edad'], bins=[-np.inf, 12, 17, 39, 59, np.inf],
    labels=['Niñez (0-12)', 'Adolescencia (13-17)', 'Adulto joven (18-39)',
            'Adulto (40-59)', 'Adulto mayor (60+)'])

resumen_actividad = (df.groupby('actividades').agg(
    participantes=('id_participante', 'count'),
    salario_mediano=('salario', 'median'),
    ganancias_medianas=('ganancias_sector', 'median'),
    con_cud=('tiene_cud', lambda s: int((s == 'Sí').sum())))
    .sort_values('salario_mediano'))
resumen_sector = (df.groupby('sectores').agg(
    participantes=('id_participante', 'count'),
    salario_mediano=('salario', 'median'),
    ganancias_medianas=('ganancias_sector', 'median'),
    con_cud=('tiene_cud', lambda s: int((s == 'Sí').sum())))
    .sort_values('salario_mediano'))

df.to_csv(LIMPIO, index=False)
print('Filas:', len(df), '| Nulos:', int(df.isna().sum().sum()))
print('Salarios imputados:', int(df['salario_imputado'].sum()),
      '| Ganancias imputadas:', int(df['ganancias_imputada'].sum()))
display(resumen_actividad.round(2))
display(resumen_sector.round(2))


## Fase 3 – Estadística descriptiva, asimetría y outliers

Comparamos los datos crudos (con los errores) contra el dataset limpio. El coeficiente de asimetría es el momento de tercer orden corregido (`Series.skew()`).


In [ ]:
def descriptivos(s, etiqueta):
    s = s.dropna()
    return {
        'dataset': etiqueta, 'n': len(s),
        'media': round(s.mean(), 2), 'mediana': round(s.median(), 2),
        'desvio': round(s.std(), 2), 'asimetria': round(s.skew(), 3),
    }

tabla = pd.DataFrame([
    descriptivos(crudo['salario'], 'salario crudo'),
    descriptivos(crudo.loc[crudo['salario'] > 0, 'salario'], 'salario sin -150.000'),
    descriptivos(df['salario'], 'salario limpio'),
    descriptivos(crudo['ganancias_sector'], 'ganancias crudas'),
    descriptivos(crudo.loc[crudo['ganancias_sector'] < 1_000_000, 'ganancias_sector'], 'ganancias sin 15M'),
    descriptivos(df['ganancias_sector'], 'ganancias limpias'),
])
display(tabla)

print('La media de ganancias cruda supera a la mediana en',
      f"{100 * (crudo['ganancias_sector'].mean() - crudo['ganancias_sector'].median()) / crudo['ganancias_sector'].median():.1f}%",
      'y su asimetría es', round(crudo['ganancias_sector'].skew(), 2),
      '-> producida por el valor 15.000.000.')


In [ ]:
def iqr_info(s, etiqueta):
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    atipicos = s[(s < lim_inf) | (s > lim_sup)]
    return {'variable': etiqueta, 'Q1': round(q1, 2), 'Q3': round(q3, 2),
            'IQR': round(iqr, 2), 'limite_inf': round(lim_inf, 2),
            'limite_sup': round(lim_sup, 2), 'atipicos': len(atipicos)}

display(pd.DataFrame([
    iqr_info(crudo['ganancias_sector'], 'ganancias crudas'),
    iqr_info(crudo['salario'], 'salario crudo'),
    iqr_info(df['ganancias_sector'], 'ganancias limpias'),
    iqr_info(df['salario'], 'salario limpio'),
]))

print('Atípicos del salario crudo:',
      sorted(crudo['salario'].dropna().tolist())[:3],
      sorted(crudo['salario'].dropna().tolist())[-3:])


### Visualizaciones (sobre datos limpios, 300 DPI)

Se guardan en `informe/` con sufijo `_limpio` para no sobrescribir las imágenes originales de la Fase 2.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.histplot(df['ganancias_sector'], kde=True, ax=axes[0], color='teal', bins=30)
axes[0].set_title('Distribución de Ganancias por Sector (datos limpios)')
axes[0].set_xlabel('Ganancias Sector')
sns.histplot(df['salario'], kde=True, ax=axes[1], color='coral', bins=30)
axes[1].set_title('Distribución de Salarios (datos limpios)')
axes[1].set_xlabel('Salario')
plt.tight_layout()
plt.savefig(FIGURAS / 'distribucion_financiera_limpia.png', dpi=300)
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='actividades', y='salario', palette='Set2')
plt.title('Diagrama de Caja de Salarios por Actividad (datos limpios)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(FIGURAS / 'boxplot_salario_actividades_limpio.png', dpi=300)
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='edad', y='salario', hue='actividades',
                palette='deep', alpha=0.8, s=70)
plt.title('Relación entre Edad y Salario según Actividad (datos limpios)')
plt.legend(title='Actividades', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIGURAS / 'scatterplot_edad_salario_limpio.png', dpi=300)
plt.show()


## Fase 4 – Conclusiones y propuestas estratégicas

### ¿Qué está pasando en el negocio?

La operación es estable en su centro: la mediana salarial es **439.541** y la de ganancias por sector **353.741**, con participantes de 5 a 74 años y cuatro actividades bien distribuidas (Natación 69, Rehabilitación 63, Hidrogimnasia 62 y Aquagym 56 registros ya deduplicados).

El problema principal es de **calidad de datos**. Un único valor de **15.000.000** infló la media de ganancias de 357.823 a 414.575 (+15,9%) y elevó el coeficiente de asimetría de 0,18 a 15,83. Hay un salario negativo (−150.000), 33 celdas vacías y dos campos con cinco variantes de escritura cada uno.

**Aquagym tiene la mediana salarial más baja (397.031)**, frente a 471.228 de Rehabilitación: un 15,7% por debajo. Por sector, Privado es el más bajo (431.986) y Educación el más alto (448.940), brecha del 3,9%. El grupo `Sin dato` (11 registros) se imputó, no se interpreta. La tenencia de CUD no explica la brecha (48,2% en Aquagym vs 60,9% en Natación).


### Estrategias propuestas

**1. Validación en origen de montos.** Reglas que rechacen o marquen `salario <= 0` y `ganancias_sector > 1.000.000`, con alerta al operador. *Impacto:* la media de ganancias vuelve de 414.575 a 357.823 y la asimetría de 15,83 a 0,18.

**2. Estandarización de catálogos e imputación estadística.** Listas desplegables cerradas para `tiene_cud` e `identidad_sexual` en el formulario de carga, e imputación con mediana por actividad. *Impacto:* se recuperan 33 celdas nulas (1,28%) y 2 valores inválidos sin salarios falsos de cero ni etiquetas fragmentadas.

**3. Revisión del mix de actividades.** Revisar precios/costos de Aquagym y rediseñar campañas de captación, usando Rehabilitación como benchmark interno. *Impacto:* ataca la brecha del 15,7% entre actividades, la señal más clara del cruce de datos.


In [ ]:
print('Resumen final')
print(' Registros limpios:', len(df))
print(' Nulos:', int(df.isna().sum().sum()))
print(' Mediana salario:', round(df['salario'].median(), 2))
print(' Mediana ganancias:', round(df['ganancias_sector'].median(), 2))
print(' Archivos generados:')
for nombre in ['distribucion_financiera_limpia.png',
               'boxplot_salario_actividades_limpio.png',
               'scatterplot_edad_salario_limpio.png']:
    ruta = FIGURAS / nombre
    print('  ', ruta.relative_to(RAIZ), 'existe:', ruta.exists())
